# Dota 2 Data Setup

- fetching hero data and matchup statistics from the OpenDota API 
- storing the data in Delta tables.

* `dota_heroes` - Hero data (name, attributes, roles)
* `dota_hero_matchups` - WR between hero pairs

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd
import requests
import time


In [0]:
BASE_URL = "https://api.opendota.com/api"
HEADERS = {"User-Agent": "Databricks-Dota2-Pipeline/1.0"}

In [0]:

def fetch_hero_metadata():
    response = requests.get(f"{BASE_URL}/heroStats", headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    heroes = []
    for h in data:
        heroes.append(
            {
                "hero_id": h["id"],
                "name": h["localized_name"],
                "primary_attr": h["primary_attr"],
                "attack_type": h["attack_type"],
                "roles": ", ".join(h["roles"]),
                "img": f"https://api.opendota.com{h['img']}",
            }
        )

    return pd.DataFrame(heroes)

def fetch_all_matchups_safe(hero_ids):
    all_matchups = []
    total_heroes = len(hero_ids)

    for idx, hero_id in enumerate(hero_ids, 1):
        url = f"{BASE_URL}/heroes/{hero_id}/matchups"
        success = False
        attempts = 0
        max_attempts = 5

        while not success and attempts < max_attempts:
            res = requests.get(url, headers=HEADERS)

            if res.status_code == 200:
                matchups = res.json()
                for m in matchups:
                    all_matchups.append(
                        {
                            "hero_id": hero_id,
                            "against_hero_id": m["hero_id"],
                            "games_played": m["games_played"],
                            "wins": m["wins"],
                        }
                    )
                success = True
                print(
                    f"[{idx}/{total_heroes}] Successfully fetched Hero ID {hero_id}"
                )

            elif res.status_code == 429:
                attempts += 1
                wait_time = attempts * 10
                print(
                    f"Rate limit hit (429) on Hero ID {hero_id}. Retrying in {wait_time}s..."
                )
                time.sleep(wait_time)

            else:
                print(
                    f"Failed hero ID {hero_id} with status {res.status_code}"
                )
                break

        time.sleep(1.1)

    df = pd.DataFrame(all_matchups)
    df["win_rate"] = (df["wins"] / df["games_played"]) * 100
    return df

In [0]:
heroes_pd = fetch_hero_metadata()

heroes_sdf = spark.createDataFrame(heroes_pd)
heroes_sdf.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("dota_heroes")

print(f"Saved {heroes_pd.shape[0]} heroes to the 'dota_heroes' table")
display(heroes_sdf)

Saved 127 heroes to 'dota_heroes' table


hero_id,name,primary_attr,attack_type,roles,img
1,Anti-Mage,agi,Melee,"Carry, Escape, Nuker",https://api.opendota.com/apps/dota2/images/dota_react/heroes/antimage.png?
2,Axe,str,Melee,"Initiator, Durable, Disabler, Carry",https://api.opendota.com/apps/dota2/images/dota_react/heroes/axe.png?
3,Bane,all,Ranged,"Support, Disabler, Nuker, Durable",https://api.opendota.com/apps/dota2/images/dota_react/heroes/bane.png?
4,Bloodseeker,agi,Melee,"Carry, Disabler, Nuker, Initiator",https://api.opendota.com/apps/dota2/images/dota_react/heroes/bloodseeker.png?
5,Crystal Maiden,int,Ranged,"Support, Disabler, Nuker",https://api.opendota.com/apps/dota2/images/dota_react/heroes/crystal_maiden.png?
6,Drow Ranger,agi,Ranged,"Carry, Disabler, Pusher",https://api.opendota.com/apps/dota2/images/dota_react/heroes/drow_ranger.png?
7,Earthshaker,str,Melee,"Support, Initiator, Disabler, Nuker",https://api.opendota.com/apps/dota2/images/dota_react/heroes/earthshaker.png?
8,Juggernaut,agi,Melee,"Carry, Pusher, Escape",https://api.opendota.com/apps/dota2/images/dota_react/heroes/juggernaut.png?
9,Mirana,agi,Ranged,"Carry, Support, Escape, Nuker, Disabler",https://api.opendota.com/apps/dota2/images/dota_react/heroes/mirana.png?
10,Morphling,agi,Ranged,"Carry, Escape, Durable, Nuker, Disabler",https://api.opendota.com/apps/dota2/images/dota_react/heroes/morphling.png?


In [0]:
hero_ids = [
    row.hero_id
    for row in spark.table("dota_heroes").select("hero_id").collect()
]

matchups_pd = fetch_all_matchups_safe(hero_ids)

matchups_sdf = spark.createDataFrame(matchups_pd)
matchups_sdf.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("dota_hero_matchups")

display(matchups_sdf.limit(10))

[1/127] Successfully fetched Hero ID 1
[2/127] Successfully fetched Hero ID 2
[3/127] Successfully fetched Hero ID 3
[4/127] Successfully fetched Hero ID 4
[5/127] Successfully fetched Hero ID 5
[6/127] Successfully fetched Hero ID 6
[7/127] Successfully fetched Hero ID 7
[8/127] Successfully fetched Hero ID 8
[9/127] Successfully fetched Hero ID 9
[10/127] Successfully fetched Hero ID 10
[11/127] Successfully fetched Hero ID 11
[12/127] Successfully fetched Hero ID 12
[13/127] Successfully fetched Hero ID 13
[14/127] Successfully fetched Hero ID 14
[15/127] Successfully fetched Hero ID 15
[16/127] Successfully fetched Hero ID 16
[17/127] Successfully fetched Hero ID 17
[18/127] Successfully fetched Hero ID 18
[19/127] Successfully fetched Hero ID 19
[20/127] Successfully fetched Hero ID 20
[21/127] Successfully fetched Hero ID 21
[22/127] Successfully fetched Hero ID 22
[23/127] Successfully fetched Hero ID 23
[24/127] Successfully fetched Hero ID 25
[25/127] Successfully fetched Hero

hero_id,against_hero_id,games_played,wins,win_rate
1,17,116,63,54.310344827586206
1,129,111,61,54.95495495495496
1,84,104,45,43.269230769230774
1,39,96,50,52.083333333333336
1,86,96,45,46.875
1,101,95,44,46.31578947368421
1,128,94,47,50.0
1,87,93,51,54.83870967741935
1,11,90,41,45.55555555555556
1,14,87,45,51.724137931034484


Checks

In [0]:
heroes_count = spark.table("dota_heroes").count()
matchups_count = spark.table("dota_hero_matchups").count()

print(f"Heroes: {heroes_count}")
print(f"Matchups: {matchups_count}")

Heroes: 127
Matchups: 15984
